 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

In [ ]:

citenum_url_link_re = re.compile(r'\[(?P<orig>\d+)\]\((?P<url>https?://[^\)]+)\)')
perplex_source_list_re = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)')
citenum_plain_re = re.compile(r'\[(?P<num>\d+)\]')

relinker = lpz.ZoteroLinkConverter()

def citenums_to_urls_dedup(num_url_pairs: list[tuple[str, str]], verbose: bool = False) -> pd.DataFrame:
    """Return a dataframe mapping from citenums to url.
    Citenums are remapped when >1 citenums map to the same URL."""
    
    url_to_citenums = defaultdict(list)
    for num, url in num_url_pairs:
        url_to_citenums[url].append(num)
    
    # Create new citation numbers if there are duplicates
    new_cite_num = 1
    lut = []
    display_citenums_map = False
    for url, nums in url_to_citenums.items():
        if (nDups := len(nums)) > 1 and verbose:
            display_citenums_map = True
            print(f'URL has {nDups} dups: {nums=}, {url=}')
            
        for num in nums:
            lut.append({'orig_num': num, 'new_num': str(new_cite_num), 'url': url})
        
        new_cite_num += 1
    
    citenums_to_url = pd.DataFrame(lut).set_index(['orig_num'])
    if display_citenums_map:
        display(citenums_to_url)
    
    return citenums_to_url

def replace_plain_body_citenum(body: str, oldnum_to_new: Dict[str, str]) -> str:
    """Replace plain citation numbers in the body with new ones."""

    return re.sub(citenum_plain_re, lambda m: f'[{oldnum_to_new[m.group("num")]}]', body)

def collect_and_fix_body_links(file_text: str, verbose: bool = False) -> tuple[str, pd.DataFrame]:
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes. """    

    section_parts = file_text.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts

    # Reassign body cite numbers if duplicate URLs are found in the sources
    source_matches = list(perplex_source_list_re.finditer(citations))
    if len(source_matches) < 1:
        print('Found no sources in file_text')
        
    citenum_url_pairs = [(match.group('num'), match.group('url')) for match in source_matches]
    citenums_to_url = citenums_to_urls_dedup(citenum_url_pairs, verbose=verbose)
        
    body_dedup = replace_plain_body_citenum(body, citenums_to_url.new_num.to_dict())
    
    return body_dedup, citenums_to_url

def make_relinks_from_source(cite_num: str, doc_url: str, all_body_cite_nums: set) -> str:
    """Returns what a relinked citation would look like if present in the body,
    given a source citation number and url.  Also appends a relinked source to
    relinked_sources."""
    
    numbered_link = f"[{cite_num}]({doc_url})"
    if zotero_item := relinker.find_zotero_item_via_url(doc_url):
        body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
        relinked_source = f'({numbered_link}) **{body_link}**'
    else:
        body_link = f"=={numbered_link}==" # mark it as "not in zotero"
        relinked_source = f'({numbered_link}) {doc_url}'
        relinked_source = f'=={relinked_source} ==' if cite_num in all_body_cite_nums else relinked_source
        
    return body_link, relinked_source


def relink_chunks(body: str, citenums_to_url: pd.DataFrame) -> tuple[str, str]:
    """Replaces body links with Zotero or Obsidian links, and returns the relinked body and sources."""

    def make_replacement_links(body: str, new_num_to_url: Dict[str, str], all_body_cite_nums: set) -> tuple[Dict[str, str], list[str]]:
        """Compute links to Zotero and Obsidian, when possible."""
        
        source_num_to_link, relinked_sources = {}, []
        for num, url in new_num_to_url.items():
            body_link, relinked_source = make_relinks_from_source(num, url, all_body_cite_nums)
            source_num_to_link[num] = body_link
            relinked_sources.append(relinked_source)
        return source_num_to_link, relinked_sources

    new_num_to_url = citenums_to_url.set_index('new_num').url.to_dict() # dedups to url

    # compute links to zotero and obsidian, when possible
    body_plain_citenums = set(re.findall(citenum_plain_re, body))
    source_num_to_link, relinked_sources = make_replacement_links(body, new_num_to_url, body_plain_citenums)
    
    body_relinked = re.sub(citenum_plain_re, 
                           lambda m: f' {source_num_to_link.get(m.group("num"))}', body)
    
    return body_relinked, relinked_sources

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    file_text = perplexity_file.read_text(encoding='utf-8')
    
    body, citenums_to_url = collect_and_fix_body_links(file_text, verbose=verbose)
    body_relinked, relinked_sources = relink_chunks(body, citenums_to_url)
    
    body_relinked = rfw.heirarch_shift_markdown_headers(body_relinked, top_level=2)
    source_link = rfw.file_link_md('source', perplexity_file)
    relinked_file.write_text(f'\n*{source_link}*\n# Response\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', encoding='utf-8')

Reading from cache.


In [3]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = True
relink_perplexity_export(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
URL has 2 dups: nums=['4', '54'], url='https://globalaffairs.org/commentary-and-analysis/blogs/brazils-systemic-mistrust-elections-and-democracy'


,new_num,url
orig_num,,
1,1,https://en.wikipedia.org/wiki/Right-wing_populism
2,2,https://www.politico.eu/article/mapped-europe-...
3,3,https://www.npr.org/2024/06/09/nx-s1-4997712/f...
4,4,https://globalaffairs.org/commentary-and-analy...
54,4,https://globalaffairs.org/commentary-and-analy...
...,...,...
77,76,https://rioonwatch.org/?p=72542
78,77,https://www.populismstudies.org/chega-emerges-...
79,78,https://www.american.edu/sis/centers/transatla...


Done.


### Test merging

In [4]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

chat_files = list(datdir.glob('*.md'))
chat_files

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [5]:
verbose = False
all_bodies, all_citenums_to_url = [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'{chat_file.stem}')

    file_text = chat_file.read_text(encoding='utf-8')
    
    body_dedup, citenums_to_url = collect_and_fix_body_links(file_text, verbose=verbose)
    all_bodies.append(body_dedup)

    citenums_to_url[['file_index','chat_file']] = file_index, chat_file
    all_citenums_to_url.append(citenums_to_url.reset_index())
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} citation numbers')

#### Make a unified cite number set for the merged document

In [6]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['new_num_int'] = df['new_num'].astype(int)

grouped = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_new_num_int=('new_num_int', 'mean')
).reset_index()

grouped = grouped.sort_values(by=['mean_file_index', 'mean_new_num_int'], 
                              ascending=True).reset_index(drop=True)

grouped['citenum_merged'] = np.arange(1, len(grouped) + 1).astype(str) # citenum == rank as sttring

# Merge back the new citenumes
df = df.merge(grouped[['url', 'citenum_merged']], on='url')

In [7]:
all_citenums_to_url = (df.sort_values('citenum_merged')
                       .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                       .drop('new_num_int', axis=1)
                       .set_index('file_index'))

all_citenums_to_url

,orig_num,doc_dedup_num,url,chat_file,new_num
file_index,,,,,
7,4,4,https://www.euronews.com/my-europe/2024/04/23/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1
2,4,4,https://www.euronews.com/my-europe/2024/04/23/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1
1,4,4,https://www.euronews.com/my-europe/2024/04/23/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1
0,4,4,https://www.euronews.com/my-europe/2024/04/23/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1
3,4,4,https://www.euronews.com/my-europe/2024/04/23/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1
...,...,...,...,...,...
5,55,54,https://www.as-coa.org/articles/explainer-braz...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,97
6,76,76,https://www.csis.org/analysis/whither-spain-ju...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,98
4,32,32,https://www.csis.org/analysis/whither-spain-ju...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,98


#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [8]:
unified_body, all_source_lines = "", []
for file_index, body_dedup in enumerate(all_bodies):
    citenums_to_url_this = all_citenums_to_url.loc[file_index]
    dedup_to_unified = citenums_to_url_this.set_index('doc_dedup_num').new_num.to_dict()

    body_unified_this = replace_plain_body_citenum(body_dedup, dedup_to_unified)
    source_link = rfw.file_link_md('source', chat_files[file_index])
    unified_body += f'# {chat_files[file_index].name}\n*{source_link}*\n\n{body_unified_this}\n'

##### Replace concantenated body links with links to Obsidian or Zotero

In [9]:
new_citenums = all_citenums_to_url[['new_num', 'url']].drop_duplicates()
new_citenums.index = new_citenums['new_num']
new_citenums.index.name = 'orig_num'
new_citenums

,new_num,url
orig_num,,
1,1,https://www.euronews.com/my-europe/2024/04/23/...
10,10,https://www.politico.eu/article/mapped-europe-...
100,100,https://thesoufancenter.org/intelbrief-2024-ma...
101,101,https://time.com/6130308/bolsonaro-brazil-2022...
102,102,https://www.politico.eu/article/far-right-euro...
...,...,...
95,95,https://www.reuters.com/world/americas/brazils...
96,96,https://www.reuters.com/world/europe/portugues...
97,97,https://www.as-coa.org/articles/explainer-braz...


In [10]:
unified_body_relinked, relinked_sources = relink_chunks(unified_body , new_citenums)

unified_body_relinked = rfw.heirarch_shift_markdown_headers(unified_body_relinked, top_level=2)
relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(citenum_plain_re, line).group(1))))

merged_output_file.write_text(f'# Responses\n{unified_body_relinked}\n# Citations\n{relinked_sources}', encoding='utf-8')

print("Done.")

Done.


## Save My Chatbot Doodling

In [ ]:
    def relink_body_source(body_text: str, sources_text: str) -> Tuple[str, str, Counter]:
        """Replace URLs with Zotero/Obsidian links in both content sections"""
        def build_source_url_to_title(sources_content: str) -> Dict[str, str]:
            """Build a dictionary mapping URLs to titles from the sources section of a "Save my Chatbot" output."""
           
            source_url_to_title = {}
            
            matches = re.findall(r'- \[(.*?)\]\((https?://\S+)\)', sources_content)
            for title, url in matches:
                normalized_url = rfw.normalize_url(url)
                title = re.sub(r'^\s*\(\d+\)\s*', '', title) # remove ref num
                source_url_to_title[normalized_url] = title.strip()
            return source_url_to_title

        def _replace_body_link(match: re.Match) -> str:
            url = rfw.normalize_url(match.group(2))
            link_num = match.group(1)
            
            if url not in source_url_to_title:
                body_link_urls_not_in_source[url] += 1
                
            if item := relinker.find_zotero_item_via_url(url):
                return relinker.create_obsidian_or_zotero_link(item)
            if title := source_url_to_title.get(url):
                if item := relinker.find_zotero_item_via_title(title):
                    return relinker.create_obsidian_or_zotero_link(item)
            
            link_nums_not_in_zotero[link_num] = True
            return f'=={match.group(0)}=='

        def _replace_source_link(match: re.Match) -> str:
            link_num, desc, url = match.groups()
            norm_url = rfw.normalize_url(url)
            item = relinker.find_zotero_item_via_url(norm_url) or relinker.find_zotero_item_via_title(desc)
            
            if item:
                new_link = relinker.create_obsidian_or_zotero_link(item)
                return f'[({link_num}) {desc}]({url}) **{new_link}**'
            if link_num in link_nums_not_in_zotero:
                return f'==[({link_num}) {desc}]({url})=='
            
            return match.group(0) # a source never used in the body

        source_url_to_title = build_source_url_to_title(sources_text)
        body_link_urls_not_in_source: Counter[str] = Counter()
        link_nums_not_in_zotero = {}

        processed_body = body_link_re.sub(_replace_body_link, body_text)
        processed_sources = source_link_re.sub(_replace_source_link, sources_text)
        
        return processed_body, processed_sources, body_link_urls_not_in_source